<a href="https://colab.research.google.com/github/1Jaffry1/student-workshop/blob/master/03_RT_DETR_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Modern Object Detection III: RT-DETR

Transformer / end-to-end object detection.


## 0. Workshop introduction

RT-DETR (Real-Time Detection Transformer) still uses a **CNN backbone**, but the rest of the pipeline is different from YOLO:

```text
YOLO:     dense predictions at many locations  +  NMS
RT-DETR:  a fixed set of object queries        +  end-to-end set prediction
```

Each **object query** is a slot that the decoder fills with at most one object. Duplicate boxes are discouraged during training (Hungarian matching), so classical NMS is not required at inference.


## 1. Learning objectives

- Describe object queries and why they replace a dense grid.
- Explain the hybrid encoder at a high level (no attention math).
- Contrast RT-DETR post-processing with YOLO’s NMS.


## How this workshop is structured

You will **not** implement neural-network layers from scratch.

The instructor cells already contain working functions for each important stage of the algorithm. Your job is to:

1. Read what each stage does and why it exists.
2. Assemble those stages in the correct order (a short coding task).
3. Change one or two parameters and watch the output change.

The demo cell is only a one-liner (`run_full_pipeline`) so you can see a result after Run all. **Do not copy that function for the assembly exercise** — wire the named stages listed in the student task.

Hands-on coding is intentionally light (~20–25% of the session). Most of the time is for understanding the pipeline.


## 2. Environment setup


In [ ]:
!pip install -q "transformers>=4.41.0,<5.0" accelerate==1.4.0 timm matplotlib opencv-python-headless pillow


In [ ]:
import platform
import sys

print("Python version:", sys.version.split()[0])
print("Platform:", platform.platform())

import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory:", round(props.total_memory / 1024 ** 3, 2), "GB")
else:
    print("GPU: None")
    print("GPU memory: n/a")
    print("\nEnable a GPU: Runtime → Change runtime type → T4 GPU, then Restart session.")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)


## Troubleshooting

| Problem | Fix |
|---|---|
| CUDA is unavailable | `Runtime → Change runtime type → T4 GPU`, then restart and Run all |
| Package import fails | `Runtime → Restart session`, then Run all |
| Checkpoint download fails | Re-run the setup / model-load cell |
| Out of memory | Use the smaller default model, or a smaller image |
| A student cell has `???` | That is expected. Fill it in, or set `RUN_STUDENT_ASSEMBLY = False` to skip it |

Do not spend workshop time debugging package conflicts. Restart and Run all first.


## 3. Imports


In [ ]:
# ==========================================
# INSTRUCTOR PROVIDED — DO NOT MODIFY
# ==========================================

import urllib.error
import urllib.request
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import patches
from PIL import Image

SAMPLE_IMAGES = {
    "bus": "https://raw.githubusercontent.com/ultralytics/ultralytics/main/ultralytics/assets/bus.jpg",
    "zidane": "https://raw.githubusercontent.com/ultralytics/ultralytics/main/ultralytics/assets/zidane.jpg",
    "cats": "http://images.cocodataset.org/val2017/000000039769.jpg",
    "living_room": "http://images.cocodataset.org/val2017/000000000139.jpg",
    "street": "http://images.cocodataset.org/val2017/000000037777.jpg",
}


def download_image(url, path="sample.jpg"):
    path = Path(path)
    if path.exists():
        return path
    req = urllib.request.Request(url, headers={"User-Agent": "object-detection-workshop/1.0"})
    try:
        with urllib.request.urlopen(req) as resp:
            path.write_bytes(resp.read())
    except urllib.error.HTTPError as err:
        loc = err.headers.get("Location")
        if err.code in (301, 302, 303, 307, 308) and loc:
            return download_image(loc, path)
        raise
    return path


def load_image(path):
    """Load an RGB uint8 image as a NumPy array (H, W, 3)."""
    return np.array(Image.open(path).convert("RGB"))


def _class_color(cls_id):
    rng = np.random.RandomState(int(cls_id) * 17 + 3)
    return rng.randint(40, 230, size=3) / 255.0


def visualize_detections(
    image,
    boxes,
    scores=None,
    labels=None,
    names=None,
    title=None,
    max_dets=60,
    prompt=None,
):
    """Draw xyxy boxes. `names` maps class id → string."""
    fig, ax = plt.subplots(1, 1, figsize=(10, 7))
    ax.imshow(image)
    ax.axis("off")
    header = title or ""
    if prompt:
        header = (header + "  |  prompt: " + str(prompt)).strip(" |")
    if header:
        ax.set_title(header, fontsize=12)

    boxes = [] if boxes is None else list(boxes)[:max_dets]
    scores = [None] * len(boxes) if scores is None else list(scores)[:max_dets]
    labels = [None] * len(boxes) if labels is None else list(labels)[:max_dets]

    for box, score, label in zip(boxes, scores, labels):
        x1, y1, x2, y2 = [float(v) for v in box]
        cls_id = 0 if label is None else int(label)
        color = _class_color(cls_id)
        ax.add_patch(
            patches.Rectangle(
                (x1, y1),
                max(x2 - x1, 1.0),
                max(y2 - y1, 1.0),
                linewidth=2,
                edgecolor=color,
                facecolor="none",
            )
        )
        name = names.get(cls_id, str(cls_id)) if isinstance(names, dict) else (str(label) if label is not None else "")
        caption = name if score is None else f"{name} {float(score):.2f}"
        ax.text(
            x1,
            max(y1 - 4, 12),
            caption,
            color="white",
            fontsize=9,
            bbox=dict(facecolor=color, edgecolor="none", pad=2, alpha=0.85),
        )
    fig.tight_layout()
    plt.show()
    return fig


## 4. Load an example image


In [ ]:
# ==========================================
# INSTRUCTOR PROVIDED — DO NOT MODIFY
# ==========================================

IMAGE_PATH = download_image(SAMPLE_IMAGES["bus"], "bus.jpg")
image = load_image(IMAGE_PATH)
plt.figure(figsize=(8, 6)); plt.imshow(image); plt.axis("off"); plt.title("Input image"); plt.show()


## 5–6. The RT-DETR pipeline

```text
Image
  → CNN backbone              multi-scale features
  → Hybrid encoder            intra-scale + cross-scale fusion
  → Object-query selection    keep the most promising encoder slots
  → Transformer decoder       refine the queries
  → Class + box predictions   one prediction per query
  → Score threshold           not NMS
```

The number of queries is fixed by the checkpoint (300 for this model). We will not retrain it; instead we visualize the highest-scoring queries.


In [ ]:
# ==========================================
# INSTRUCTOR PROVIDED — DO NOT MODIFY
# ==========================================

from transformers import AutoImageProcessor, AutoModelForObjectDetection

CHECKPOINT = "PekingU/rtdetr_r18vd"  # small ResNet-18 variant for Colab
processor = AutoImageProcessor.from_pretrained(CHECKPOINT)
rtdetr = AutoModelForObjectDetection.from_pretrained(CHECKPOINT).to(DEVICE).eval()
RTDETR_NAMES = dict(rtdetr.config.id2label)
print("Loaded", CHECKPOINT, "| queries:", rtdetr.config.num_queries, "| classes:", len(RTDETR_NAMES))


def preprocess_image(image):
    """Resize/normalize the way the RT-DETR checkpoint expects (typically 640×640)."""
    inputs = processor(images=image, return_tensors="pt")
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    if "pixel_mask" not in inputs:
        b, _, h, w = inputs["pixel_values"].shape
        inputs["pixel_mask"] = torch.ones((b, h, w), device=DEVICE)
    return inputs


@torch.no_grad()
def extract_multiscale_features(inputs):
    """CNN backbone: last three feature scales."""
    pixel_values = inputs["pixel_values"]
    pixel_mask = inputs.get("pixel_mask")
    if pixel_mask is None:
        b, _, h, w = pixel_values.shape
        pixel_mask = torch.ones((b, h, w), device=pixel_values.device)
    features = rtdetr.model.backbone(pixel_values, pixel_mask)
    shapes = [tuple(src.shape) for src, _mask in features]
    print("backbone scales:", shapes)
    return features


@torch.no_grad()
def encode_features(features):
    """Hybrid encoder: intra-scale attention + cross-scale CNN fusion."""
    proj = [rtdetr.model.encoder_input_proj[i](src) for i, (src, _mask) in enumerate(features)]
    encoded = rtdetr.model.encoder(proj)
    return encoded


@torch.no_grad()
def select_queries_and_decode(inputs):
    """Query selection + Transformer decoder + class/box heads.

    These stages are tightly coupled in the official implementation, so they are
    wrapped together rather than faked as independent layers.
    """
    outputs = rtdetr(**inputs)
    return outputs


def postprocess_predictions(outputs, image, threshold=0.3):
    """Score threshold only. No NMS — RT-DETR is trained as a set predictor."""
    h, w = image.shape[:2]
    target_sizes = torch.tensor([(h, w)], device=outputs.logits.device)
    result = processor.post_process_object_detection(
        outputs, threshold=threshold, target_sizes=target_sizes
    )[0]
    return {
        "boxes": result["boxes"].detach().cpu(),
        "scores": result["scores"].detach().cpu(),
        "labels": result["labels"].detach().cpu(),
        "names": RTDETR_NAMES,
        "n_queries": outputs.logits.shape[1],
    }


def query_preview(outputs, image, top_k=30):
    """Visualize the highest-scoring object queries *before* the confidence cut."""
    logits = outputs.logits[0]
    boxes = outputs.pred_boxes[0]
    prob = logits.sigmoid()
    scores, labels = prob.max(-1)
    topk = torch.topk(scores, k=min(top_k, scores.numel()))
    h, w = image.shape[:2]
    cxcywh = boxes[topk.indices].detach().cpu()
    xyxy = torch.stack(
        [
            (cxcywh[:, 0] - cxcywh[:, 2] / 2) * w,
            (cxcywh[:, 1] - cxcywh[:, 3] / 2) * h,
            (cxcywh[:, 0] + cxcywh[:, 2] / 2) * w,
            (cxcywh[:, 1] + cxcywh[:, 3] / 2) * h,
        ],
        dim=1,
    )
    return {
        "boxes": xyxy,
        "scores": scores[topk.indices].detach().cpu(),
        "labels": labels[topk.indices].detach().cpu(),
        "names": RTDETR_NAMES,
    }


def show_dets(image, dets, title=None):
    visualize_detections(
        image,
        dets["boxes"].numpy(),
        dets["scores"].numpy(),
        dets["labels"].numpy(),
        names=dets["names"],
        title=title,
    )



print("RT-DETR helpers ready.")


## 7. Student assembly


In [ ]:
# ==========================================
# STUDENT TASK
# ==========================================
# Set True after you replace every ??? with the correct function call.
RUN_STUDENT_ASSEMBLY = False

if RUN_STUDENT_ASSEMBLY:
    inputs = preprocess_image(image)
    features = ???          # extract_multiscale_features
    encoded = ???           # encode_features  (inspects the encoder; decoder reruns internally)
    outputs = ???           # select_queries_and_decode
    detections = postprocess_predictions(outputs, image, threshold=0.3)
    show_dets(image, detections, title="Student assembly")
    print("queries:", detections["n_queries"], "| detections:", len(detections["boxes"]))
else:
    print('Skipping student assembly. The instructor demo above already ran the pipeline.')
    print('During the exercise: fill in the TODOs, then set RUN_STUDENT_ASSEMBLY = True.')


## 8–9. Experiments


Change THRESHOLD & TOP_QUERIES and rerun.

## 10. Think about it

1. What replaces YOLO’s dense grid + NMS in RT-DETR?
2. Why is a confidence **threshold** still useful even without NMS?
3. The checkpoint uses 300 queries. What would go wrong if a scene had more than 300 objects?

The number of queries is baked into the pretrained weights, so we do not change it here.


## 11. Optional challenge

Run the same image through YOLO (previous notebook) and RT-DETR. Where do they disagree?


## 12. Summary

- RT-DETR = CNN features + Transformer decoder + **object queries**.
- Post-processing is a score cut, not NMS.

Next: **YOLO-World** — the class list is no longer fixed; it comes from **text**.
